In [13]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
from utils.read_txtfile import get_data_fromtxt


data = get_data_fromtxt(path=r'D:\Projects\Assignment\LLM & AI Assistant Projects\Voice-Gen\tools\playlist.txt')

In [3]:
data

{'topics': {'knowledge': {'authors': [{'name': 'Vui Vẻ',
     'urls': ['https://www.youtube.com/watch?v=en6RQgjAEmM',
      'https://www.youtube.com/watch?v=X5YqZh4ysyY',
      'https://www.youtube.com/watch?v=99LXQSYzH-8']},
    {'name': 'Spiderum',
     'urls': ['https://www.youtube.com/watch?v=fc8xseBKtiQ']}]},
  'philosophy': {'authors': [{'name': 'Duy Thanh Nguyen',
     'urls': ['https://www.youtube.com/watch?v=_9TO4Vog26I',
      'https://www.youtube.com/watch?v=O-fBnUWDORc']}]},
  'self-development': {'authors': []}}}

In [9]:
type(data)

dict

In [7]:
topic = 'knowledge'

In [13]:
for item in data['topics'][f"{topic}"]['authors']:
    for url in item['urls']:
        print(url)
        print(item['name'])
        

https://www.youtube.com/watch?v=en6RQgjAEmM
Vui Vẻ
https://www.youtube.com/watch?v=X5YqZh4ysyY
Vui Vẻ
https://www.youtube.com/watch?v=99LXQSYzH-8
Vui Vẻ
https://www.youtube.com/watch?v=fc8xseBKtiQ
Spiderum


# Test âm thanh 

In [4]:
import os
os.environ["PATH"] += r";C:\ffmpeg\ffmpeg-8.1-essentials_build\bin"

In [ ]:
from IPython.display import Audio
Audio(r"D:\Projects\Assignment\LLM & AI Assistant Projects\Voice-Gen\data\raw\knowledge\Vui Vẻ\Toan_b_l_ch_s_th_gi_i_trong_8_phut.wav")

# Understanding pathlib 

In [ ]:
import os

os.chdir('..') # step back parent folde

In [16]:
from pathlib import Path
from tqdm import tqdm

raw_wav_dir = Path("data/raw/knowledge")
authors  = list(raw_wav_dir.glob("*"))
print(raw_wav_dir.resolve())

D:\Projects\Assignment\LLM & AI Assistant Projects\Voice-Gen\data\raw\knowledge


In [13]:
list(authors)

[WindowsPath('data/raw/knowledge/Spiderum'),
 WindowsPath('data/raw/knowledge/Vui Vẻ')]

In [20]:
for author in tqdm(authors, desc="Getting a author list"):
    files = list(author.glob("*.wav")) 
    for file in tqdm(files, desc="Processing Audio"):
        print(file)

Processing Audio: 100%|██████████| 1/1 [00:00<?, ?it/s]it/s]


data\raw\knowledge\Spiderum\Nikola_Tesla_-_Bi_k_ch_cu_c_i_nha_phat_minh_thien_tai_Vi_t_Cung_Ti_u_Hy_Th_Gi_i.wav


Getting a author list: 100%|██████████| 2/2 [00:00<00:00, 137.94it/s]

data\raw\knowledge\Vui Vẻ\Toan_b_l_ch_s_th_gi_i_trong_8_phut.wav
data\raw\knowledge\Vui Vẻ\T_t_c_cac_ki_u_gi_c_m_trong_10_phut.wav
data\raw\knowledge\Vui Vẻ\T_t_t_n_t_t_cac_b_nh_lien_quan_n_r_ng_trong_12_phut.wav


# Check cuda

In [4]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

2.5.1+cu121
True
NVIDIA GeForce GTX 1660 Ti


In [25]:
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader , Dataset
import torch.nn.functional as F


class SimpleNN(nn.Module):
    def __init__(self, input_shape):
        super().__init__()
        self.layer1 = nn.Linear(input_shape, 100)
        self.layer2 = nn.Linear(100, 2)

    def forward(self, x):
        x = F.relu(self.layer1(x))   
        x = self.layer2(x)
        return x

In [ ]:
# data 
X = torch.rand(size =(100 , 10))
y = torch.concat([torch.ones(50) , torch.zeros(50)])

class Mydataset(Dataset):
    def __init__(self , X , y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index] , self.y[index]
    
dataset = Mydataset(X , y)
dataloader = DataLoader(dataset, batch_size=16 , shuffle= True)

In [32]:
model = SimpleNN(input_shape=10)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model.parameters(), lr = 4e-4 , momentum=0.8)

def train_one_epoch(dataloader , model , criterion ,optimizer , device = 'cuda'):
    total_loss = 0.0

    for batch_X, batch_y in tqdm(dataloader):

        batch_X = batch_X.to(device)
        batch_y = batch_y.long().to(device)

        out = model(batch_X)
        loss = criterion(out , batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    
    return total_loss / len(dataloader)

def train(dataloader , model , criterion ,optimizer , epochs = 10,  device = 'cuda'):
    model.to(device)
    for i in range(epochs):

        loss = train_one_epoch(dataloader , model , criterion ,optimizer)

        print(f"[ Epoch {i} - Loss = {loss} ]")

In [33]:
train(dataloader, model , criterion , optimizer)

100%|██████████| 7/7 [00:00<00:00, 19.19it/s]


[ Epoch 0 - Loss = 0.691137032849448 ]


100%|██████████| 7/7 [00:00<00:00, 466.60it/s]


[ Epoch 1 - Loss = 0.6929640599659511 ]


100%|██████████| 7/7 [00:00<00:00, 500.33it/s]


...

[ Epoch 7 - Loss = 0.691112790788923 ]


100%|██████████| 7/7 [00:00<00:00, 285.37it/s]


[ Epoch 8 - Loss = 0.6912133182798114 ]


100%|██████████| 7/7 [00:00<00:00, 233.35it/s]

[ Epoch 9 - Loss = 0.6893235785620553 ]
